# Data Cleaning & Quality Assessment

## Overview
1. Data Loading and Initial Inspection  
2. Duplicate Records (_id)  
3. Nested Field Normalization (spending_behavior)  
4. Data Quality Discovery (completeness, consistency, validity)  
5. Data Remediation (Cleaning Pipeline)  
6. Final Validation & Export  

## 1. Data Loading and Initial Inspection

This section loads the raw JSON file, flattens it into a tabular format, and performs basic inspection to understand schema and potential data quality issues.

**Cleaning Approach:**
We first stabilize the dataset structure by removing duplicate application records and normalizing nested fields.
After the dataset is fully tabular, we profile data quality issues (missing values, inconsistent formats, invalid values)
and apply a transparent, rule-based remediation pipeline.

In [ ]:
# Imports and Data Loading

import json
import numpy as np
import pandas as pd

# Adjust pandas display settings so wide DataFrames remain readable in the notebook
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# Path to the raw dataset (JSON format)
DATA_PATH = "../data/raw_credit_applications.json"

# Load the JSON file into Python as a list of dictionaries
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

# Convert the nested JSON structure into a flat tabular DataFrame
df = pd.json_normalize(raw)

# Basic inspection of the loaded dataset
print("Loaded records:", len(raw))      # number of applications in the raw JSON
print("DataFrame shape:", df.shape)     # (rows, columns) after normalization
display(df.head(3))                     # preview first 3 rows

Loaded records: 502
DataFrame shape: (502, 21)


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN


**Initial data inspection** shows a total of 502 application records and 21 attributes.  
Several fields exhibit substantial missingness (e.g., processing_timestamp, loan_purpose, notes), while others are fully populated.  
Data types are heterogeneous, with multiple numeric attributes stored as object types, indicating potential consistency issues to be addressed in subsequent steps.

## 2. Duplicate Recoders (_id)

In [6]:
# Duplicate Records (_id)

# Ensure that the unique identifier column exists in the dataset
# If not, stop execution because duplicate detection relies on this field
if "_id" not in df.columns:
    raise KeyError("Expected column '_id' not found. Check your input JSON structure.")

# Identify rows that share the same _id (potential duplicate applications)
# keep=False marks ALL rows involved in duplication
dup_mask = df.duplicated(subset=["_id"], keep=False)

# Count how many rows are part of duplicate groups
dup_rows = int(dup_mask.sum())

print("Rows with duplicate _id:", dup_rows)

# If duplicates exist, display the most frequent duplicate IDs
if dup_rows > 0:
    display(
        df.loc[dup_mask, ["_id"]]      # select rows that contain duplicate IDs
        .value_counts()                # count occurrences of each ID
        .rename("count")               # rename the count column for clarity
        .reset_index()                 # convert the result into a DataFrame
        .sort_values("count", ascending=False)  # show most repeated IDs first
        .head(10)                      # limit output to top 10 duplicates
    )

# Remove duplicate records, keeping the first occurrence of each _id
before = len(df)
df = df.drop_duplicates(subset=["_id"], keep="first").copy()
after = len(df)

# Report how many rows were removed during deduplication
print("Before dedup:", before)
print("After dedup :", after)
print("Dropped     :", before - after)

Rows with duplicate _id: 4


,_id,count
0,app_042,2
1,app_001,2


Before dedup: 502
After dedup : 500
Dropped     : 2


Duplicate application records were assessed using the application identifier.  
Hard duplicates were quantified and removed to ensure record uniqueness prior to further data quality remediation.

## 3. Nested Field Normalization (spending_behavior)


The original dataset contains a nested list field (`spending_behavior`) with category–amount pairs.
This structure is flattened into a single wide table by creating one column per spending category
(`spending_<category>`). After normalization, the dataset contains one row per application and no
nested fields remain.

In [ ]:
# Flatten spending_behavior into separate spending_* columns

# Check whether the nested spending_behavior field exists
if "spending_behavior" not in df.columns:
    print("No 'spending_behavior' column found. Skipping.")
else:
    # Turn each list entry in spending_behavior into its own temporary row
    tmp = df[["_id", "spending_behavior"]].explode("spending_behavior")

    # Extract the spending category and amount from each dictionary entry intp seperate fields
    tmp["category"] = tmp["spending_behavior"].apply(
        lambda x: x.get("category") if isinstance(x, dict) else np.nan
    )
    tmp["amount"] = tmp["spending_behavior"].apply(
        lambda x: x.get("amount") if isinstance(x, dict) else np.nan
    )

    # Convert amount to numeric so calculations work correctly
    tmp["amount"] = pd.to_numeric(tmp["amount"], errors="coerce")

    # Reshape to wide format: one row per application, one column per spending category
    wide = (
        tmp.pivot_table(
            index="_id",
            columns="category",
            values="amount",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    # Clean category names so they become valid and readable column names
    def safe_spending_col(name):
        name = str(name).strip().replace(" ", "_")
        name = "".join(ch for ch in name if ch.isalnum() or ch == "_")
        return f"spending_{name}"

    wide = wide.rename(columns={c: safe_spending_col(c) for c in wide.columns if c != "_id"})

    # Remove the original nested column and merge the new flat spending columns back into the dataset
    df = df.drop(columns=["spending_behavior"]).merge(wide, on="_id", how="left")

    print("Flattened spending_behavior into columns.")
    print("New shape:", df.shape)

# Final check: confirm that no columns still contain nested list/dict values
nested_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, (list, dict))).any()]
print("Nested columns remaining:", nested_cols)

# Preview the updated dataset
display(df.head(3))

Flattened spending_behavior into columns.
New shape: (500, 35)
Nested columns remaining: []


,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spending_Adult_Entertainment,spending_Alcohol,spending_Dining,spending_Education,spending_Entertainment,spending_Fitness,spending_Gambling,spending_Groceries,spending_Healthcare,spending_Insurance,spending_Rent,spending_Shopping,spending_Transportation,spending_Travel,spending_Utilities
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0


The nested spending_behavior field was flattened by converting each spending category into its own column (e.g., spending_food, spending_travel). This increased the dataset from 21 to 35 columns, creating 14 new spending-related features and removing all nested structures from the data.

## 4. Data Quality Discovery

After normalizing all nested fields, we assess data quality issues across four dimensions:

- **Completeness**: missing and hidden missing values
- **Consistency**: formatting and data type inconsistencies
- **Validity**: impossible or out-of-range values
- **Accuracy**: cross-field logic violations and plausibility checks (proxy for ground-truth accuracy)
- **Uniqueness**: potential duplicate identifiers (e.g. SSN)

This step is purely diagnostic: no data is modified.

In [ ]:
# Data Quality Discovery (NO MODIFICATIONS)
# This section identifies potential data quality issues without modifying the dataset.

print("=== COMPLETENESS ===")

# Define common tokens that may represent missing values but are not stored as NaN
MISSING_TOKENS = {"", " ", "na", "n/a", "nan", "null", "none", "-", "--"}

# Count standard missing values (NaN) per column
standard_nulls = df.isna().sum()

# Detect hidden missing values stored as strings (e.g., "na", "-", etc.)
hidden_nulls = df.map(
    lambda x: isinstance(x, str) and x.strip().lower() in MISSING_TOKENS
).sum()

# Combine both types of missing values into a summary table
total_records = len(df)
missing_df = pd.DataFrame({
    "Standard Nulls": standard_nulls,
    "Hidden Nulls": hidden_nulls,
    "Total Missing": standard_nulls + hidden_nulls
})

# Add percentage of total records for each column
missing_df["% of Records"] = (missing_df["Total Missing"] / total_records * 100).round(1).astype(str) + "%"

# Display columns that contain any missing values
display(
    missing_df[missing_df["Total Missing"] > 0]
    .sort_values("Total Missing", ascending=False)
)

print("\n=== CONSISTENCY (DATA TYPES & CATEGORIES) ===")

# Show columns stored as 'object' type, which may indicate inconsistent data types
print("Object-type columns (potential inconsistencies):")
display(df.dtypes[df.dtypes == "object"])

# Inspect categorical fields to detect inconsistent labels or formatting
for col in ["applicant_info.gender", "decision.rejection_reason"]:
    if col in df.columns:
        print(f"\nUnique values in {col}:")
        display(pd.Series(df[col].dropna().unique()))

print("\n=== VALIDITY (NUMERIC RANGES) ===")

# Review numeric columns to identify unrealistic values (e.g., negative balances)
numeric_cols = df.select_dtypes(include=[np.number]).columns
display(df[numeric_cols].describe().T[["min", "max", "mean"]])

# Quantify specific invalid values
invalid_credit = int((df["financials.credit_history_months"] < 0).sum()) if "financials.credit_history_months" in df.columns else 0
invalid_savings = int((df["financials.savings_balance"] < 0).sum()) if "financials.savings_balance" in df.columns else 0
print(f"\nNegative credit_history_months: {invalid_credit} records ({invalid_credit / total_records * 100:.1f}%)")
print(f"Negative savings_balance:       {invalid_savings} record  ({invalid_savings / total_records * 100:.1f}%)")

print("\n=== UNIQUENESS (SSN) ===")

# Check if the SSN field contains duplicate values across multiple records
if "applicant_info.ssn" in df.columns:
    dup_ssn = df[df.duplicated(subset=["applicant_info.ssn"], keep=False)]
    n_dup_ssn_records = len(dup_ssn)
    print(
        f"Duplicate SSNs: {dup_ssn['applicant_info.ssn'].nunique()} "
        f"shared across {n_dup_ssn_records} records "
        f"({n_dup_ssn_records / total_records * 100:.1f}%)"
    )

In [ ]:
# === ACCURACY (Cross-field Logic & Plausibility Checks) ===
#
# Accuracy asks: even if a value is present and correctly formatted,
# does it correctly reflect the real-world fact?
#
# Without an external ground truth we cannot verify accuracy directly.
# Instead we apply two proxy approaches:
#   1. Cross-field logic checks — fields that contradict each other within
#      the same record indicate that at least one value must be wrong.
#   2. Plausibility bounds — values that are valid in format but implausible
#      given domain knowledge (e.g. a credit applicant under 18).

print("=== ACCURACY (CROSS-FIELD LOGIC CHECKS) ===")

total_records = len(df)

# --- Check 1: Approved application that also carries a rejection reason ---
# An application cannot be both approved and have a rejection reason.
# If both are present, one of the two fields is inaccurate.
if {"decision.loan_approved", "decision.rejection_reason"}.issubset(df.columns):
    acc1 = int(
        (df["decision.loan_approved"] == True) & df["decision.rejection_reason"].notna()
    ).sum() if False else int(
        ((df["decision.loan_approved"] == True) & (df["decision.rejection_reason"].notna())).sum()
    )
    print(f"Approved + rejection_reason present:      {acc1} records ({acc1 / total_records * 100:.1f}%)")

# --- Check 2: Rejected application that carries an approved_amount ---
# A rejected application should not have an approved loan amount.
if {"decision.loan_approved", "decision.approved_amount"}.issubset(df.columns):
    income_col = pd.to_numeric(df["decision.approved_amount"], errors="coerce")
    acc2 = int(((df["decision.loan_approved"] == False) & income_col.notna()).sum())
    print(f"Rejected  + approved_amount present:      {acc2} records ({acc2 / total_records * 100:.1f}%)")

# --- Check 3: Rejected application that carries an interest rate ---
if {"decision.loan_approved", "decision.interest_rate"}.issubset(df.columns):
    rate_col = pd.to_numeric(df["decision.interest_rate"], errors="coerce")
    acc3 = int(((df["decision.loan_approved"] == False) & rate_col.notna()).sum())
    print(f"Rejected  + interest_rate present:        {acc3} records ({acc3 / total_records * 100:.1f}%)")

print("\n=== ACCURACY (PLAUSIBILITY CHECKS) ===")

# --- Check 4: Debt-to-income ratio above 1.0 ---
# A DTI > 1.0 means total debt obligations exceed gross annual income,
# which is implausible as a stable financial state. Flagged as a likely
# data entry or calculation error.
if "financials.debt_to_income" in df.columns:
    dti = pd.to_numeric(df["financials.debt_to_income"], errors="coerce")
    acc4 = int((dti > 1.0).sum())
    print(f"debt_to_income > 1.0 (implausible):       {acc4} records ({acc4 / total_records * 100:.1f}%)")

# --- Check 5: Applicant age below 18 at time of application ---
# Credit applicants must be adults. A date_of_birth implying age < 18
# indicates either a data entry error or a policy violation.
if "applicant_info.date_of_birth" in df.columns:
    dob_parsed = pd.to_datetime(df["applicant_info.date_of_birth"], errors="coerce", dayfirst=True)
    age_years = (pd.Timestamp("today") - dob_parsed).dt.days / 365.25
    acc5 = int((age_years < 18).sum())
    print(f"Applicant age < 18 (ineligible):          {acc5} records ({acc5 / total_records * 100:.1f}%)")

print("\nNote: zero violations on cross-field checks confirm internal decision-field")
print("consistency, but do not rule out inaccuracies in the underlying input values.")

## Data Quality Findings

### Completeness (n = 500 records)
| Field | Missing Count | % of Records | Note |
|---|---|---|---|
| `notes` | 500 | 100.0% | Entirely empty field |
| `loan_purpose` | 450 | 90.0% | Mostly unpopulated |
| `processing_timestamp` | 438 | 87.6% | Mostly unpopulated |
| `decision.rejection_reason` | 292 | 58.4% | Expected: absent for approved cases |
| `decision.approved_amount` / `decision.interest_rate` | 208 | 41.6% | Expected: absent for rejected cases |
| `applicant_info.email` (hidden, blank strings) | 7 | 1.4% | Stored as `""` not `NaN` |
| `financials.annual_income` | 5 | 1.0% | Backfilled from `annual_salary` |
| `applicant_info.ssn` | 4 | 0.8% | |
| `applicant_info.ip_address` | 4 | 0.8% | |
| `applicant_info.date_of_birth` (hidden, blank strings) | 4 | 0.8% | Stored as `""` not `NaN` |
| `applicant_info.gender` (hidden, blank strings) | 2 | 0.4% | Stored as `""` not `NaN` |
| `applicant_info.zip_code` (hidden, blank strings) | 1 | 0.2% | Stored as `""` not `NaN` |

### Consistency
- **`financials.annual_income`** stored as `object` dtype instead of numeric, affecting all records.
- **`applicant_info.gender`** encoded inconsistently: `Male`, `M`, `Female`, `F`, and empty string all present in the same column.

### Validity
| Issue | Count | % of Records |
|---|---|---|
| Negative `financials.credit_history_months` (min = -10) | 2 | 0.4% |
| Negative `financials.savings_balance` (min = -5,000) | 1 | 0.2% |
| `financials.debt_to_income` > 1.0 (max = 1.85) | see Accuracy below | N/A |
| Malformed email addresses | 11 | 2.2% |

### Accuracy
Accuracy asks whether a value correctly reflects the real-world fact it represents, not just whether it is present or well-formatted. Without an external ground truth, we apply two proxy approaches: **cross-field logic checks** (fields within the same record that contradict each other) and **plausibility bounds** (values that are format-valid but domain-implausible).

**Cross-field logic checks** (see code cell above for counts):
- *Approved application with a `rejection_reason` present*: logically contradictory, one field must be wrong.
- *Rejected application with `approved_amount` or `interest_rate` present*: logically contradictory.

**Plausibility checks:**
- *`debt_to_income` > 1.0*: a ratio above 1.0 means total debt obligations exceed gross income, which is implausible as a steady financial state and likely indicates a calculation or entry error.
- *Applicant age < 18*: credit applicants must be adults; any DOB implying age under 18 is either an entry error or a policy violation.

### Uniqueness
- **2 duplicate `_id` values** (app_001, app_042), 4 rows total (0.8%), 2 records removed after dedup.
- **2 SSN values** duplicated across **8 records (1.6%)**, flagged for review (not auto-dropped).

- **High missingness** in optional/context fields: `notes` (500) and `processing_timestamp` (438).
- **Misaligned income fields**: `financials.annual_salary` is mostly missing (495), while `financials.annual_income` exists but is stored as `object` (needs numeric casting).
- **Categorical gaps**: `loan_purpose` has many missing values (450). `applicant_info.gender` contains inconsistent encodings (`Male`, `Female`, `M`, `F`, empty).
- **Decision-field missingness is conditional**: `decision.approved_amount` and `decision.interest_rate` are missing in 208 records (likely rejected cases).
- **Invalid values detected**: negative `financials.credit_history_months` and negative `financials.savings_balance` exist and must be handled.
- **Uniqueness issue**: 2 SSNs appear across 8 records (flag for review; do not automatically drop without a clear policy).

### Format Validity Checks (Regex-Based)

In addition to missing and range-based checks, we validate the **format correctness**
of selected identifier fields (email, SSN) using regular expressions.
This step identifies malformed values without modifying the dataset.

In [9]:
# Format Validity Checks (Discovery Only)
# Validate whether email and SSN fields follow expected formats using regex.

import re

print("=== FORMAT VALIDITY (REGEX CHECKS) ===")

# ---- Email format validation ----
EMAIL_REGEX = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

if "applicant_info.email" in df.columns:
    # Clean email values before validation
    emails = df["applicant_info.email"].dropna().astype(str).str.strip()

    # Flag emails that do not match the expected format
    malformed_email_mask = ~emails.str.match(EMAIL_REGEX)

    print(f"Found {malformed_email_mask.sum()} malformed emails.")
    display(emails[malformed_email_mask].head(10))


# ---- SSN format validation ----
# Expected US SSN format: 123-45-6789
SSN_REGEX = r"^\d{3}-\d{2}-\d{4}$"

if "applicant_info.ssn" in df.columns:
    ssns = df["applicant_info.ssn"].dropna().astype(str).str.strip()

    # Flag SSNs that do not match the expected format
    malformed_ssn_mask = ~ssns.str.match(SSN_REGEX)

    print(f"Found {malformed_ssn_mask.sum()} malformed SSNs.")

=== FORMAT VALIDITY (REGEX CHECKS) ===
Found 11 malformed emails.


26                           
138    mike johnson@gmail.com
181     test.user.outlook.com
187                          
275                          
276          john.doe@invalid
297                          
298                          
368              sarah.smith@
447                          
Name: applicant_info.email, dtype: str

Found 0 malformed SSNs.


## 5. Data Remediation (Cleaning Pipeline)

We now apply a rule-based cleaning pipeline to address the issues identified in discovery:

- **Consistency**: standardize `gender`, cast data types, consolidate salary into income
- **Validity**: convert impossible negative values to missing (`NaN`)
- **Completeness**: fill missing categorical fields with `Unknown` where appropriate
- **Format validity**: invalidate malformed emails (set to `NaN`)
- **Uniqueness**: flag duplicate SSNs for review (do not drop automatically)

In [11]:
# Data Remediation (Cleaning Pipeline)
# Apply rule-based fixes to address the issues found during the discovery step.

df_clean = df.copy()
remediation = {}  # track how many values/rows were affected by each fix

# --- Normalize hidden-missing tokens (e.g., "na", "-", "") into real NaN values ---
MISSING_TOKENS = {"", " ", "na", "n/a", "nan", "null", "none", "-", "--"}

def normalize_missing(x):
    if x is None:
        return np.nan
    if isinstance(x, float) and np.isnan(x):
        return np.nan
    if isinstance(x, str) and x.strip().lower() in MISSING_TOKENS:
        return np.nan
    return x

df_clean = df_clean.map(normalize_missing)
remediation["normalized_missing_tokens"] = "done"

# --- Fix income fields: cast annual_income to numeric and backfill from annual_salary if present ---
if "financials.annual_income" in df_clean.columns:
    df_clean["financials.annual_income"] = pd.to_numeric(df_clean["financials.annual_income"], errors="coerce")
else:
    df_clean["financials.annual_income"] = np.nan

if "financials.annual_salary" in df_clean.columns:
    missing_before = int(df_clean["financials.annual_income"].isna().sum())
    df_clean["financials.annual_income"] = df_clean["financials.annual_income"].fillna(df_clean["financials.annual_salary"])
    df_clean = df_clean.drop(columns=["financials.annual_salary"])
    remediation["merged_salary_into_income_missing_before"] = missing_before

# --- Ensure key numeric fields are stored as numeric types ---
num_cols = [
    "financials.credit_history_months",
    "financials.debt_to_income",
    "financials.savings_balance",
    "decision.approved_amount",
    "decision.interest_rate",
]
for col in num_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# --- Parse date_of_birth into a proper datetime (invalid formats become NaT) ---
if "applicant_info.date_of_birth" in df_clean.columns:
    df_clean["applicant_info.date_of_birth"] = pd.to_datetime(
        df_clean["applicant_info.date_of_birth"], errors="coerce", dayfirst=True
    )
    remediation["parsed_date_of_birth"] = "done"

# --- Standardize gender labels and fill missing values ---
if "applicant_info.gender" in df_clean.columns:
    df_clean["applicant_info.gender"] = (
        df_clean["applicant_info.gender"]
        .astype("string")
        .str.strip()
        .replace({"M": "Male", "F": "Female", "m": "Male", "f": "Female"})
    )
    n_missing_gender = int(df_clean["applicant_info.gender"].isna().sum())
    df_clean["applicant_info.gender"] = df_clean["applicant_info.gender"].fillna("Unknown")
    remediation["filled_gender_unknown"] = n_missing_gender

# --- Fill missing loan purpose with an explicit placeholder category ---
if "loan_purpose" in df_clean.columns:
    n_missing_purpose = int(df_clean["loan_purpose"].isna().sum())
    df_clean["loan_purpose"] = df_clean["loan_purpose"].fillna("Unknown")
    remediation["filled_loan_purpose_unknown"] = n_missing_purpose

# --- Invalidate malformed emails (keep valid ones, set invalid formats to NaN) ---
EMAIL_REGEX = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

if "applicant_info.email" in df_clean.columns:
    df_clean["applicant_info.email"] = df_clean["applicant_info.email"].astype("string").str.strip()

    email_series = df_clean["applicant_info.email"].dropna()
    malformed_mask = ~email_series.str.match(EMAIL_REGEX)

    remediation["malformed_emails_set_to_nan"] = int(malformed_mask.sum())
    df_clean.loc[email_series.index[malformed_mask], "applicant_info.email"] = np.nan

# --- Set logically invalid negative values to NaN (cannot be negative) ---
if "financials.savings_balance" in df_clean.columns:
    bad = df_clean["financials.savings_balance"] < 0
    remediation["savings_negative_to_nan"] = int(bad.sum())
    df_clean.loc[bad, "financials.savings_balance"] = np.nan

if "financials.credit_history_months" in df_clean.columns:
    bad = df_clean["financials.credit_history_months"] < 0
    remediation["credit_history_negative_to_nan"] = int(bad.sum())
    df_clean.loc[bad, "financials.credit_history_months"] = np.nan

# --- Enforce decision logic consistency between approval flag and related fields ---
if "decision.loan_approved" in df_clean.columns:
    approved = df_clean["decision.loan_approved"] == True
    rejected = df_clean["decision.loan_approved"] == False

    if "decision.rejection_reason" in df_clean.columns:
        bad = approved & df_clean["decision.rejection_reason"].notna()
        remediation["cleared_rejection_reason_for_approved"] = int(bad.sum())
        df_clean.loc[bad, "decision.rejection_reason"] = np.nan

        miss = rejected & df_clean["decision.rejection_reason"].isna()
        remediation["filled_rejection_reason_unknown_for_rejected"] = int(miss.sum())
        df_clean.loc[miss, "decision.rejection_reason"] = "Unknown"

    for c in ["decision.approved_amount", "decision.interest_rate"]:
        if c in df_clean.columns:
            bad = rejected & df_clean[c].notna()
            remediation[f"cleared_{c}_for_rejected"] = int(bad.sum())
            df_clean.loc[bad, c] = np.nan

# --- Flag duplicate SSNs for review (do not drop automatically) ---
if "applicant_info.ssn" in df_clean.columns:
    dup_ssn = df_clean["applicant_info.ssn"].notna() & df_clean.duplicated(subset=["applicant_info.ssn"], keep=False)
    df_clean["flag_duplicate_ssn"] = dup_ssn
    remediation["flag_duplicate_ssn_records"] = int(dup_ssn.sum())

# Report what was changed
display(pd.DataFrame([remediation]).T.rename(columns={0: "count_or_note"}))
print("Shape after remediation:", df_clean.shape)
display(df_clean.head(3))

,count_or_note
normalized_missing_tokens,done
merged_salary_into_income_missing_before,5
parsed_date_of_birth,done
filled_gender_unknown,2
filled_loan_purpose_unknown,450
malformed_emails_set_to_nan,4
savings_negative_to_nan,1
credit_history_negative_to_nan,2
cleared_rejection_reason_for_approved,0
filled_rejection_reason_unknown_for_rejected,0


Shape after remediation: (500, 35)


,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,notes,spending_Adult_Entertainment,spending_Alcohol,spending_Dining,spending_Education,spending_Entertainment,spending_Fitness,spending_Gambling,spending_Groceries,spending_Healthcare,spending_Insurance,spending_Rent,spending_Shopping,spending_Transportation,spending_Travel,spending_Utilities,flag_duplicate_ssn
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-09-03,10036,73000.0,23.0,0.20,31212.0,False,algorithm_risk_score,Unknown,NaN,NaN,NaN,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0,False
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,NaT,10032,78000.0,51.0,0.18,17915.0,False,algorithm_risk_score,Unknown,NaN,NaN,NaN,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0,False
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,NaT,10075,61000.0,41.0,0.21,37909.0,True,NaN,vacation,3.7,59000.0,NaN,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0,False


## 6. Final Validation & Export

We perform a final sanity check on the cleaned dataset (shape, remaining missingness, basic validity signals),
then export the cleaned dataset for downstream analysis.

In [12]:
# Final Validation & Export
# Quick sanity checks to confirm remediation worked as intended.

print("Final shape:", df_clean.shape)

# Show the columns that still have missing values (top 10 by count)
missing_final = df_clean.isna().sum().sort_values(ascending=False)
missing_final = missing_final[missing_final > 0]
print("\nTop missing columns after remediation:")
display(missing_final.head(10))

# Confirm key remediation outcomes
print("\nChecks:")
print(" - annual_salary present?:", "financials.annual_salary" in df_clean.columns)
print(
    " - duplicate SSN flagged records:",
    int(df_clean.get("flag_duplicate_ssn", pd.Series(False)).sum())
)

Final shape: (500, 35)

Top missing columns after remediation:


notes                               500
processing_timestamp                438
applicant_info.date_of_birth        357
decision.rejection_reason           292
decision.approved_amount            208
decision.interest_rate              208
applicant_info.email                 11
applicant_info.ssn                    4
applicant_info.ip_address             4
financials.credit_history_months      2
dtype: int64


Checks:
 - annual_salary present?: False
 - duplicate SSN flagged records: 4


In [13]:
# Export
# Save the cleaned dataset to the processed data folder for downstream use.

OUT_CSV = "../data/processed/cleaned_credit_applications.csv"

# Write the cleaned DataFrame to a CSV file (without the index column)
df_clean.to_csv(OUT_CSV, index=False)

# Confirm export location
print("\nExported:")
print(" -", OUT_CSV)


Exported:
 - ../data/processed/cleaned_credit_applications.csv
